In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline


In [2]:
# Load trained model and encoders from Part 2
with open('../models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/label_encoders.pkl', 'rb') as f:
    le_dict = pickle.load(f)

with open('../models/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

# Load cleaned data
df = pd.read_csv('../data/raw/StudentsPerformance.csv')
df.columns = df.columns.str.replace(' ', '_').str.replace('/', '_')
df['average_score'] = df[['math_score', 'reading_score', 'writing_score']].mean(axis=1)
df['at_risk'] = (df['average_score'] < 60).astype(int)

print(f"Dataset loaded: {df.shape}")
print(f"At-risk students: {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)")

Dataset loaded: (1000, 10)
At-risk students: 285 (28.5%)


In [3]:
from langchain_core.documents import Document

# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
documents.append(Document(page_content=overall_stats, metadata={"source": "overall_stats"}))

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
    Completing test prep strongly reduces at-risk probability.
    Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
    Standard lunch (higher SES) significantly reduces risk.
    Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support reviewZ
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
documents.append(Document(page_content=overall_stats, metadata={"source": "overall_stats"}))

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
    Completing test prep strongly reduces at-risk probability.
    Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
    Standard lunch (higher SES) significantly reduces risk.
    Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support review
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
  Completing test prep strongly reduces at-risk probability.
  Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
  Standard lunch (higher SES) significantly reduces risk.
  Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support review
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

Knowledge base built: 6 documents
Knowledge base built: 6 documents
Knowledge base built: 11 documents


In [4]:
# Use sentence-transformers for embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS vector store from documents
vectorstore = FAISS.from_documents(documents, embeddings)

# Save it so you don't rebuild every time
vectorstore.save_local("../models/faiss_index")

print("Vector store created and saved.")
print(f"Indexed {vectorstore.index.ntotal} document chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created and saved.
Indexed 11 document chunks


In [5]:
# Retriever — fetches top 3 relevant documents for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def analyst_agent(question):
    """
    Answers questions about the student dataset
    by retrieving relevant knowledge base documents.
    """
    # Retrieve relevant docs
    relevant_docs = retriever.get_relevant_documents(question)
    
    # Build context from retrieved docs
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Generate answer grounded in retrieved context
    answer = f"""
ANALYST AGENT — EduPulse
========================
Question: {question}

Retrieved Context:
{context}

Answer based on EduPulse data:
"""
    # Add interpretation layer
    question_lower = question.lower()
    
    if "risk" in question_lower and "factor" in question_lower:
        answer += "The top risk factors identified by the ML model (confirmed by SHAP analysis) are:\n1. Not completing test preparation course\n2. Free/reduced lunch (socioeconomic indicator)\n3. Lower parental education level"
    
    elif "how many" in question_lower or "number" in question_lower:
        at_risk_count = df['at_risk'].sum()
        total = len(df)
        answer += f"There are {at_risk_count} at-risk students out of {total} total ({at_risk_count/total*100:.1f}%)."
    
    elif "test prep" in question_lower or "preparation" in question_lower:
        completed = df[df['test_preparation_course']=='completed']['average_score'].mean()
        none = df[df['test_preparation_course']=='none']['average_score'].mean()
        answer += f"Students who completed test prep scored {completed:.1f} on average vs {none:.1f} for those who didn't — a {abs(completed-none):.1f} point difference."
    
    elif "lunch" in question_lower or "socioeconomic" in question_lower:
        standard = df[df['lunch']=='standard']['average_score'].mean()
        reduced = df[df['lunch']=='free/reduced']['average_score'].mean()
        answer += f"Students with standard lunch scored {standard:.1f} on average vs {reduced:.1f} for free/reduced lunch — a {abs(standard-reduced):.1f} point gap."
    
    else:
        answer += context[:500] + "..."
    
    return answer

# Test the analyst agent
question = "What are the main risk factors for student performance?"
# Retriever — fetches top 3 relevant documents for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def analyst_agent(question):
    """
    Answers questions about the student dataset
    by retrieving relevant knowledge base documents.
    """
    # Retrieve relevant docs
    if hasattr(retriever, "get_relevant_documents"):
        relevant_docs = retriever.get_relevant_documents(question)
    else:
        relevant_docs = vectorstore.similarity_search(question, k=3)
    
    # Build context from retrieved docs
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Generate answer grounded in retrieved context
    answer = f"""
ANALYST AGENT — EduPulse
========================
Question: {question}

Retrieved Context:
{context}

Answer based on EduPulse data:
"""
    # Add interpretation layer
    question_lower = question.lower()
    
    if "risk" in question_lower and "factor" in question_lower:
        answer += "The top risk factors identified by the ML model (confirmed by SHAP analysis) are:\n1. Not completing test preparation course\n2. Free/reduced lunch (socioeconomic indicator)\n3. Lower parental education level"
    
    elif "how many" in question_lower or "number" in question_lower:
        at_risk_count = df['at_risk'].sum()
        total = len(df)
        answer += f"There are {at_risk_count} at-risk students out of {total} total ({at_risk_count/total*100:.1f}%)."
    
    elif "test prep" in question_lower or "preparation" in question_lower:
        completed = df[df['test_preparation_course']=='completed']['average_score'].mean()
        none = df[df['test_preparation_course']=='none']['average_score'].mean()
        answer += f"Students who completed test prep scored {completed:.1f} on average vs {none:.1f} for those who didn't — a {abs(completed-none):.1f} point difference."
    
    elif "lunch" in question_lower or "socioeconomic" in question_lower:
        standard = df[df['lunch']=='standard']['average_score'].mean()
        reduced = df[df['lunch']=='free/reduced']['average_score'].mean()
        answer += f"Students with standard lunch scored {standard:.1f} on average vs {reduced:.1f} for free/reduced lunch — a {abs(standard-reduced):.1f} point gap."
    
    else:
        answer += context[:500] + "..."
    
    return answer

# Test the analyst agent
question = "What are the main risk factors for student performance?"
print(analyst_agent(question))


ANALYST AGENT — EduPulse
Question: What are the main risk factors for student performance?

Retrieved Context:

Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score 72.7
- Students with no test prep: avg score 65.0
- Difference: 7.6 points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.



Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score 72.7
- Students with no test prep: avg score 65.0
- Difference: 7.6 points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.



Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score 70.8
- Students with free/reduced lunch: a

In [6]:
def intervention_agent(student_data: dict):
    """
    Takes a student's profile, runs ML prediction,
    and generates a personalised intervention plan
    grounded in SHAP findings.
    
    student_data: dict with keys matching feature_cols
    Example: {
        'gender': 'female',
        'race_ethnicity': 'group B', 
        'parental_level_of_education': 'some high school',
        'lunch': 'free/reduced',
        'test_preparation_course': 'none'
    }
    """
    # Encode student data
    encoded = {}
    for col in feature_cols:
        val = student_data.get(col)
        if col in le_dict:
            try:
                encoded[col] = le_dict[col].transform([val])[0]
            except:
                encoded[col] = 0
        else:
            encoded[col] = val
    
    # Run ML prediction
    input_df = pd.DataFrame([encoded])
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]
    
    # Build intervention plan
    plan = f"""
INTERVENTION AGENT — EduPulse
==============================
Student Profile: {student_data}
Risk Prediction: {'⚠️ AT RISK' if prediction == 1 else '✅ NOT AT RISK'}
Risk Probability: {probability*100:.1f}%

"""
    
    if prediction == 1:
        plan += "PERSONALISED INTERVENTION PLAN:\n"
        plan += "-" * 40 + "\n"
        
        # Test prep recommendation
        if student_data.get('test_preparation_course') == 'none':
            plan += """
📚 PRIORITY 1 — Test Preparation Enrollment
   Action: Enroll student in test preparation course before next assessment
   Rationale: Test prep is the #1 predictor of at-risk status (SHAP analysis)
   Expected impact: 5-10 point average score improvement
   Timeline: Immediate
"""
        
        # Socioeconomic recommendation
        if student_data.get('lunch') == 'free/reduced':
            plan += """
🤝 PRIORITY 2 — Socioeconomic Support
   Action: Flag for financial support review + welfare services
   Rationale: Lunch type (SES proxy) is the 2nd strongest risk predictor
   Steps: Connect with peer mentor, increase check-in to weekly, assign tutoring
   Timeline: This week
"""
        
        # Parental education recommendation
        low_edu = ['some high school', 'high school']
        if student_data.get('parental_level_of_education') in low_edu:
            plan += """
👨‍👩‍👧 PRIORITY 3 — Family Engagement
   Action: Increase parental communication + college readiness support
   Rationale: Lower parental education correlates with higher academic risk
   Steps: Send weekly progress reports, assign academic counselor
   Timeline: Begin this month
"""
        
        # Combined high-risk flag
        if (student_data.get('test_preparation_course') == 'none' and 
            student_data.get('lunch') == 'free/reduced'):
            plan += """
🚨 HIGH PRIORITY CASE — Multiple Risk Factors Present
   This student has both top risk factors active simultaneously.
   Escalate to both academic and welfare support teams immediately.
"""
    else:
        plan += f"This student is not currently at risk (risk probability: {probability*100:.1f}%).\n"
        plan += "Continue monitoring. Schedule standard check-in at next assessment cycle."
    
    return plan

# Test with a high-risk student profile
high_risk_student = {
    'gender': 'female',
    'race_ethnicity': 'group B',
    'parental_level_of_education': 'some high school',
    'lunch': 'free/reduced',
    'test_preparation_course': 'none'
}

print(intervention_agent(high_risk_student))



INTERVENTION AGENT — EduPulse
Student Profile: {'gender': 'female', 'race_ethnicity': 'group B', 'parental_level_of_education': 'some high school', 'lunch': 'free/reduced', 'test_preparation_course': 'none'}
Risk Prediction: ⚠️ AT RISK
Risk Probability: 54.9%

PERSONALISED INTERVENTION PLAN:
----------------------------------------

📚 PRIORITY 1 — Test Preparation Enrollment
   Action: Enroll student in test preparation course before next assessment
   Rationale: Test prep is the #1 predictor of at-risk status (SHAP analysis)
   Expected impact: 5-10 point average score improvement
   Timeline: Immediate

🤝 PRIORITY 2 — Socioeconomic Support
   Action: Flag for financial support review + welfare services
   Rationale: Lunch type (SES proxy) is the 2nd strongest risk predictor
   Steps: Connect with peer mentor, increase check-in to weekly, assign tutoring
   Timeline: This week

👨‍👩‍👧 PRIORITY 3 — Family Engagement
   Action: Increase parental communication + college readiness support

In [7]:
print("=" * 60)
print("EDUPULSE AI SYSTEM — DEMO")
print("=" * 60)

# Analyst Agent queries
questions = [
    "What are the top risk factors for students?",
    "How many students are currently at risk?",
    "What is the impact of test preparation on scores?",
    "Which socioeconomic factors affect student performance?"
]

print("\n--- ANALYST AGENT ---")
for q in questions:
    print(f"\nQ: {q}")
    response = analyst_agent(q)
    # Print just the answer part
    print(response.split("Answer based on EduPulse data:")[-1].strip())
    print("-" * 40)

# Intervention Agent
print("\n--- INTERVENTION AGENT ---")
low_risk_student = {
    'gender': 'male',
    'race_ethnicity': 'group A',
    'parental_level_of_education': "bachelor's degree",
    'lunch': 'standard',
    'test_preparation_course': 'completed'
}

print("\nLow-risk student:")
print(intervention_agent(low_risk_student))

print("\nHigh-risk student:")
print(intervention_agent(high_risk_student))


EDUPULSE AI SYSTEM — DEMO

--- ANALYST AGENT ---

Q: What are the top risk factors for students?
The top risk factors identified by the ML model (confirmed by SHAP analysis) are:
1. Not completing test preparation course
2. Free/reduced lunch (socioeconomic indicator)
3. Lower parental education level
----------------------------------------

Q: How many students are currently at risk?
There are 285 at-risk students out of 1000 total (28.5%).
----------------------------------------

Q: What is the impact of test preparation on scores?
Students who completed test prep scored 72.7 on average vs 65.0 for those who didn't — a 7.6 point difference.
----------------------------------------

Q: Which socioeconomic factors affect student performance?
Students with standard lunch scored 70.8 on average vs 62.2 for free/reduced lunch — a 8.6 point gap.
----------------------------------------

--- INTERVENTION AGENT ---

Low-risk student:

INTERVENTION AGENT — EduPulse
Student Profile: {'gender

In [8]:
# Save vectorstore (already saved in Cell 4)
# Save a summary of what the system can do

summary = {
    'knowledge_base_docs': len(documents),
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'vector_store': 'FAISS',
    'ml_model': 'Logistic Regression (recall-optimised)',
    'top_risk_factors': ['test_preparation_course', 'lunch', 'parental_level_of_education'],
    'agents': ['Analyst Agent', 'Intervention Agent']
}

import json
with open('../models/system_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("System saved.")
print(json.dumps(summary, indent=2))

System saved.
{
  "knowledge_base_docs": 11,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "vector_store": "FAISS",
  "ml_model": "Logistic Regression (recall-optimised)",
  "top_risk_factors": [
    "test_preparation_course",
    "lunch",
    "parental_level_of_education"
  ],
  "agents": [
    "Analyst Agent",
    "Intervention Agent"
  ]
}


In [9]:
# ## Part 3 — System Output Summary

# ### Analyst Agent Findings
# | Question | Answer |
# |---|---|
# | At-risk students | 285 out of 1000 (28.5%) |
# | Test prep score gap | 7.6 points (completed vs none) |
# | Lunch/SES score gap | 8.6 points (standard vs free/reduced) |
# | Top risk factors | test_preparation_course, lunch, parental_level_of_education |

### Key Insight
# Socioeconomic factors (lunch type, 8.6pt gap) have a slightly larger 
# raw impact on scores than academic preparation (test prep, 7.6pt gap).
# This suggests welfare support should be prioritised alongside 
# academic intervention for high-risk students.

### Intervention Agent
# Successfully generates personalised, SHAP-grounded intervention plans.
# High-risk profile (no test prep + free/reduced lunch) correctly 
# triggers multi-team escalation with three prioritised action items.